<a href="https://colab.research.google.com/github/juliene-pearl/FUNDAI-Laboratories-Ysolana/blob/main/Lab4_Logic_KR_Ysolana.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: Logic and Knowledge Representation

## Fundamentals of Artificial Intelligence

**Name:** Juliene Pearl D. Ysolana
**Course:** BSCSAI
**Section:** 2A 09282
**Date:** September 15, 2026

**GitHub URL:** https://github.com/juliene-pearl/FUNDAI-Laboratories-Ysolana

## Description
This laboratory uses Python and SymPy to perform truth table generation, satisfiability checking, theorem proving, and logical deduction.



In [36]:
from sympy import symbols, And, Or, Not, Implies, Equivalent, satisfiable
from itertools import product

In [37]:
P, Q, R = symbols('P Q R')

In [38]:
def print_truth_table(expression, symbol_list):
    header = [str(s) for s in symbol_list] + [str(expression)]
    print(" | ".join(header))
    print("-" * (5 * len(header)))

    for values in product([True, False], repeat=len(symbol_list)):
        mapping = dict(zip(symbol_list, values))
        result = bool(expression.subs(mapping))
        row = [str(v) for v in values] + [str(result)]
        print(" | ".join(row))

    print()

print("Negation: NOT P")
print_truth_table(Not(P), [P])

print("Conjunction: P AND Q")
print_truth_table(And(P, Q), [P, Q])

print("Disjunction: P OR Q")
print_truth_table(Or(P, Q), [P, Q])

print("Implication: P -> Q")
print_truth_table(Implies(P, Q), [P, Q])

print("Biconditional: P <-> Q")
print_truth_table(Equivalent(P, Q), [P, Q])

Negation: NOT P
P | ~P
----------
True | False
False | True

Conjunction: P AND Q
P | Q | P & Q
---------------
True | True | True
True | False | False
False | True | False
False | False | False

Disjunction: P OR Q
P | Q | P | Q
---------------
True | True | True
True | False | True
False | True | True
False | False | False

Implication: P -> Q
P | Q | Implies(P, Q)
---------------
True | True | True
True | False | False
False | True | True
False | False | True

Biconditional: P <-> Q
P | Q | Equivalent(P, Q)
---------------
True | True | True
True | False | False
False | True | False
False | False | True



In [39]:
def is_tautology(expression, symbol_list):
    for values in product([False, True], repeat=len(symbol_list)):
        mapping = dict(zip(symbol_list, values))
        if not bool (expression.subs(mapping)):
            return False

    return True

law_of_excluded_middle = Or(P, Not(P))
contradiction = And(P, Not(P))
simple_implication = Implies(P, Q)

print("P OR NOT P is a tautology:", is_tautology(law_of_excluded_middle, [P]))
print("P AND NOT P is a tautulogy:", is_tautology(contradiction, [P]))
print("P -> Q is a tautology:", is_tautology(simple_implication, [P, Q]))

P OR NOT P is a tautology: True
P AND NOT P is a tautulogy: False
P -> Q is a tautology: False


In [40]:
def is_satisfiable(expression):
    return satisfiable(expression) is not False

print("P AND NOT Q is satisfiable:", is_satisfiable(And(P, Not(P))))
print("P OR Q is satisfiable:", is_satisfiable(Or(P, Q)))
print("P -> Q isa satisfiable:", is_satisfiable(Implies(P, Q)))

P AND NOT Q is satisfiable: False
P OR Q is satisfiable: True
P -> Q isa satisfiable: True


In [41]:
def to_conjunction(kb):
    if isinstance(kb, list):
        return And(*kb)
    return kb

def kb_entails(kb, conclusion):
    kb_expression = to_conjunction(kb)
    counter_check = And(kb_expression, Not(conclusion))
    return satisfiable(counter_check) is False


In [42]:
def check_entailment(kb, conclusion, label="Query"):
    holds = kb_entails(kb, conclusion)
    kb_expression = to_conjunction(kb)
    counterexample = satisfiable(And(kb_expression, Not(conclusion)))

    print(label)

    if holds:
        print("Result: Entailment holds.")
    else:
        print("Result: Entailment does not hold.")
        print("Counterexample model:", counterexample)

    print("-" * 60)
    return holds

In [43]:
Rain, Wet = symbols('Rain Wet')

kb_rain = [
    Implies(Rain, Wet),
    Rain
]

check_entailment(kb_rain, Wet, "Theorem Proving: Rain Example")

Theorem Proving: Rain Example
Result: Entailment holds.
------------------------------------------------------------


True

In [44]:
kb_invalid = [
    Implies(Rain, Wet),
    Wet
]

check_entailment(kb_invalid, Rain, "Invalid Inference: Affirming the consequent")

Invalid Inference: Affirming the consequent
Result: Entailment does not hold.
Counterexample model: {Wet: True, Rain: False}
------------------------------------------------------------


False

In [45]:
print("Logical Deduction Rules")
print("=" * 60)

# Modus Ponens
check_entailment(
    [P, Implies(P, Q)],
    Q,
    "Modus Ponens: P, P -> Q, therefore Q"
)

# Modus Tollens
check_entailment(
    [Not(Q), Implies(P, Q)],
    Not(P),
    "Modus Tollens: NOT Q, P -> Q, therefore NOT P"
)

Logical Deduction Rules
Modus Ponens: P, P -> Q, therefore Q
Result: Entailment holds.
------------------------------------------------------------
Modus Tollens: NOT Q, P -> Q, therefore NOT P
Result: Entailment holds.
------------------------------------------------------------


True

## Grounded First-Order Logic Example

Full First-Order Logic includes objects and quantifiers.
For this laboratory, we demonsrate a simple grounded FOL example by converting FOL atoms into propositional symbols.

English:
- All humans are mortal.
- Socrates is human.
- Therefore, Socrates is mortal.

Grounded propositiona form:
- Human_Socrates -> Mortal_Socrates

In [46]:
Human_Socrates, Mortal_Socrates = symbols('Human_Socrates Mortal_Socrates')

kb_Socrates = [
    Implies(Human_Socrates, Mortal_Socrates),
    Human_Socrates
]

check_entailment(
    kb_Socrates,
    Mortal_Socrates,
    "Grounded FOL: Socrates is mortal"
)

Grounded FOL: Socrates is mortal
Result: Entailment holds.
------------------------------------------------------------


True

In [47]:
def make_human_mortal_kb(constants):
    kb = []
    human = {}
    mortal = {}

    for name in constants:
        h, m = symbols(f'Human_{name} Mortal_{name}')
        human[name] = h
        mortal[name] = m
        kb.append(Implies(h,m))

    return kb, human, mortal

In [48]:
constants = ["Socrates", "Plato"]

kb_people, human, mortal = make_human_mortal_kb(constants)

# Add facts
kb_people.append(human["Socrates"])
kb_people.append(human["Plato"])

# Query: Is Plato mortal?
check_entailment(
    kb_people,
    mortal["Plato"],
    "Grounded FOL with multiple constants: Is Plato Mortal?"
)

Grounded FOL with multiple constants: Is Plato Mortal?
Result: Entailment holds.
------------------------------------------------------------


True

## Guide Questions and Answers

### 1. What is the difference between syntax and semantics?
**Answer:** [Syntax refers to the legal constructon of sentences in Logic, using symbols and languages. Semantics is the interpretation and how we read those symbols]

### 2. Why is 'P -> Q' true when 'P' is false?
**Answer:** [Because a conditional statement is only considered false when a true condition fails to produce a true result]

### 3. What does it mean for a knowledge base to entail a conclusion?
**Answer:** [For a knowledge base to entail a conclusion, it means that whenever all the statements (facts and rules) in the knowledge base are true, the conclusion must also be true.]

### 4. How does theorem proving use satisfiability checking?
**Answer:** [Theorem proving uses satisfiability checking by reducing the task of proving a conjecture is valid to the task of showing that its negation is unsatisfiable.]

### 5. What is one limitation of propositional logic compares to First-Order Logic?
**Answer:** [One major limitation of propositional logic compared to First-Order Logic is its lack of expressive power regarding objects, properties, and relations.]

## Reflection

### Challenges Encountered
- [I had difficulties in trying to understand on how First-Order Logic is used and how this content is correlated to the other parts of the code]

### What I learned
- [In this laboratory exercise, I explored the practical implementation of Logic and Knowledge Representation using Python and the `sympy` library. Through this experiment, I learned how abstract logical rules can be programmatically modeled, executed, and validated by a machine.]